<a href="https://colab.research.google.com/github/lsgrep/agents/blob/claude/agent-building-lessons-16749f/notebooks/08_fanout.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 8 — Fan-out, and when it pays

**The claim you should be able to make when you finish:** *"Fan-out is a context
play, not a speed play. It divides the quadratic by n and multiplies or divides
your failure probability depending on a topology choice most designs never state
out loud."*

Multi-agent is usually justified as parallelism. That is the least interesting
reason and often not even true — wall-clock is bounded by the slowest branch and
by your rate limits.

The real argument falls out of lab 2, and the real cost falls out of lab 3.
Twenty minutes.

In [ ]:
# Cell 1 — bootstrap. No GPU, no API key, no spend.
REPO, BRANCH = "https://github.com/lsgrep/agents.git", "claude/agent-building-lessons-16749f"

import os, subprocess, sys

if not os.path.isdir("agents"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO], check=True)
else:
    subprocess.run(["git", "-C", "agents", "pull", "--ff-only", "-q"], check=False)
sys.path.insert(0, os.path.abspath("agents"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'matplotlib', 'numpy'], check=True)

import agentlab
env = agentlab.notebook_setup()

## 1. The arithmetic

One agent doing `m` turns bills roughly `m*P0 + g*m²/2`.

Split that work across `n` subagents doing `m/n` turns each. **Each has its own
transcript**, so:

```
n * [ (m/n)*P0 + g*(m/n)²/2 ]  =  m*P0 + g*m²/(2n)
```

The quadratic term is divided by `n`. The fixed prefix term is unchanged. Then
reality adds overhead: each subagent carries its own system prompt and schemas,
the orchestrator writes a brief for each, and it reads a report back from each —
and those reports land in the orchestrator's transcript, where lab 2's rules
apply to them all over again.

In [ ]:
from agentlab.budget import LoopShape
from agentlab.multiagent import derive_fanout

shape = LoopShape(system_tokens=1_200, tool_tokens=6_000, prompt_tokens=300,
                  assistant_tokens=250, result_tokens=1_400, turns=48)

print(derive_fanout(shape, n_workers=4, p_worker=0.95, conjunctive=True))

Tokens: 2.2M down to 840K. Peak context per agent: 26K instead of 87K.

Reliability: **95% down to 81%**, and the check fires. That is the trade nobody
puts on the architecture diagram.

## 2. Topology decides the sign

Fan-out has two shapes and they have opposite reliability. Same boxes, same
arrows, opposite outcome.

- **Conjunctive** — the task is *split*, every branch must succeed:
  `p ** n`. Fan-out **multiplies** failure.
- **Disjunctive** — the same question is attacked several ways, any branch will
  do: `1 - (1-p) ** n`. Fan-out **divides** failure.

In [ ]:
from agentlab.multiagent import conjunctive_success, disjunctive_success

print(f"{'workers':>8} {'conjunctive (all)':>19} {'disjunctive (any)':>19}")
for n in (1, 2, 3, 5, 8, 12):
    print(f"{n:>8} {conjunctive_success(0.95, n):>18.1%} {disjunctive_success(0.95, n):>18.4%}")

Twelve 95% workers, all required: a **54%** system. Twelve 95% workers where any
one suffices: **99.99999%**.

If you take one thing from this lab, take the habit of asking, out loud, in a
design review: *"does every branch have to work, or does any one of them?"* It
is most of the review.

### The uncomfortable corollary

Most real fan-outs are conjunctive, because splitting work is the obvious reason
to fan out. So the default multi-agent design **reduces** reliability, and the
fix is not more agents — it is making each branch independently verifiable, so a
failed branch can be detected and retried without re-running the whole thing.
That is lab 3's verifier argument, arriving from a different direction.

## 3. Why token cost is the wrong metric

Sweep the number of workers on tokens alone and the answer is always "more".

In [ ]:
from agentlab.multiagent import optimal_fanout

result = optimal_fanout(shape, p_worker=0.95, conjunctive=True)
print(f"{'n':>3} {'tokens':>12} {'$':>8} {'P(ok)':>8} {'$/success':>11}")
for r in result["rows"]:
    mark = ""
    if r["n"] == result["cheapest_n"]:
        mark += "  <- cheapest tokens"
    if r["n"] == result["best_n"]:
        mark += "  <- best cost per success"
    print(f"{r['n']:>3} {r['total']:>12,} {r['usd']:>8.2f} {r['p_success']:>8.1%} "
          f"{r['usd_per_success']:>11.2f}{mark}")

Tokens fall monotonically. **Cost per success does not** — past the optimum you
are buying cheaper runs that fail more often, and paying for the retries.

`usd / P(success)` is the metric. It has an interior optimum and it is usually a
small number: single digits, not dozens.

There is a second bound that is not economic at all. Work stops being divisible.
Forty-eight dependent steps do not become forty-eight independent agents, and a
model that says otherwise is pricing an architecture nobody can build.

In [ ]:
print(f"{'min turns per worker':>21} {'max workers':>12}")
for minimum in (2, 4, 8, 12, 24):
    print(f"{minimum:>21} {optimal_fanout(shape, max_workers=48, min_turns_per_worker=minimum)['max_n_divisible']:>12}")

## 4. The multiplier you should expect

Production deep-research systems report roughly an order of magnitude more
tokens than a single-agent chat for the same user-facing question. Given the
algebra above says fan-out is *cheaper* per unit of work, that looks like a
contradiction. It is not.

Subagents do far more **total exploration** because they can. The context
ceiling that limited one agent no longer binds, so the system reads more, tries
more branches, and produces a better answer. You are not paying a tax on the
split. **You are buying breadth with money.**

Which means the honest way to present a fan-out proposal is not "this will be
cheaper" — it usually will not be, at the same quality — but "this buys breadth
that a single context cannot hold, at roughly this multiple, and here is the
class of question where that is worth it."

In [ ]:
from agentlab.multiagent import cost, fanout_tokens, single_agent_tokens

one = single_agent_tokens(shape)
print(f"one agent, 48 turns: {one:,} tok  (${cost(one):.2f})\n")
print(f"{'exploration':>12} {'total work':>11} {'tokens':>13} {'$':>9} {'vs 1 agent':>11}")
for factor in (1, 2, 4, 8, 12):
    wide = shape.at_turns(shape.turns * factor)
    total = fanout_tokens(wide, 5)["total"]
    print(f"{f'{factor}x':>12} {wide.turns:>8} turns {total:>13,} {cost(total):>9.2f} "
          f"{total / one:>10.1f}x")
print("\nThe reported ~10x multiplier corresponds to roughly 5-8x more exploration,")
print("not to the split being wasteful. You are buying breadth, and it is a")
print("different answer — not the same answer, cheaper.")

## 5. The checklist

Fan-out pays when the work is **breadth-first over independent branches** and
the results are **small relative to what was read** to produce them. It fails
when subagents must coordinate: they cannot see each other's transcripts, so
shared mutable state turns into two agents confidently doing contradictory
things.

In [ ]:
from agentlab.multiagent import when_to_fan_out

scenarios = [
    ("research 8 competitors, then summarise",
     dict(read_heavy=True, parallelisable=True, shared_state=False, turns=40, budget_multiplier=10)),
    ("refactor a module, step by step",
     dict(read_heavy=False, parallelisable=False, shared_state=True, turns=40)),
    ("answer one question from one doc",
     dict(read_heavy=True, parallelisable=True, shared_state=False, turns=4)),
    ("migrate 200 files, each independent",
     dict(read_heavy=False, parallelisable=True, shared_state=False, turns=60, budget_multiplier=5)),
]
for name, kw in scenarios:
    verdict = when_to_fan_out(**kw)
    print(f"{name:<40} {'FAN OUT' if verdict['fan_out'] else 'single agent'}")
    for reason in verdict["reasons"]:
        print(f"{'':<42} - {reason}")

The third one is worth noting: it fails the check on turns alone. A four-turn
task has no quadratic to divide, so fan-out is pure overhead — and this is
easily the most common unnecessary multi-agent design.

The fourth is the case fan-out is genuinely for, even though it is not
read-heavy: 200 independent files is a disjunctive-*ish* structure where each
branch is verifiable on its own, so a failed branch is retried rather than
sinking the run.

## 6. The failure that is specific to orchestration

One thing has no analogue in a single agent: **a subagent that does not come
back**. Times out, returns malformed output, hits a rate limit, or fails its tool
call.

The orchestrator has to decide: retry, fail over, return a partial result, or
abort. That decision needs to be *designed*, and the default — waiting, then
silently synthesising from whatever arrived — produces confident answers with
holes in them, which is the worst available outcome.

In [ ]:
print(f"{'workers'} {'':<4}{'0 fail':>9}{'1 fails':>9}{'2 fail':>9}   partial-answer quality")
for n in (3, 5, 8):
    p_all = conjunctive_success(0.95, n)
    p_one = n * 0.05 * (0.95 ** (n - 1))
    p_two = (n * (n - 1) / 2) * (0.05 ** 2) * (0.95 ** (n - 2))
    print(f"{n:>7} {'':<4}{p_all:>9.1%}{p_one:>9.1%}{p_two:>9.1%}   "
          f"{'answer is missing ' + str(round(100/n))}% of its evidence")
print("\nAt 8 workers you lose at least one branch a third of the time. If the")
print("orchestrator does not say so in its answer, the answer is quietly wrong.")

## What you can now say

- *"Fan-out is a context play, not a speed play — it divides the quadratic by
  n because each subagent has its own transcript."*
- *"Five 95% subagents that all have to work is a 77% system. Conjunctive
  fan-out multiplies failure; disjunctive divides it."*
- *"We optimise cost per success, not cost. Token cost alone always says 'more
  workers'."*
- *"The 10x token multiplier isn't the split being inefficient — it's buying
  breadth a single context can't hold."*
- *"Subagents can't see each other's transcripts, so shared mutable state is the
  failure mode."*
- *"At 8 workers you lose a branch a third of the time, so the orchestrator has
  to say what's missing."*

## Next

**[Lab 9](09_security.ipynb)** — the attack that follows directly from
everything above.